# 01 – Exploratory Data Analysis: Road Damage Detection

Übersicht über Datensatz, Annotationen und erste Modellergebnisse.

## 1 · Setup & Imports

In [ ]:
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
from ultralytics import YOLO

# ── Constants ──────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
MODEL_PATH    = Path('../models/best.pt')
DATASET_YAML  = '../dataset.yaml'

CLASS_NAMES   = {0: 'Pothole', 1: 'Crack', 2: 'Manhole'}
CLASS_COLORS  = {'Pothole': '#E05C5C', 'Crack': '#5C9BE0', 'Manhole': '#5CBE7A'}
SPLITS        = ['train', 'val', 'test']
SEED          = 42

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Setup complete.')

## 2 · Dataset-Übersicht

In [ ]:
# ── Bilder pro Split ───────────────────────────────────────────────────────
split_sizes: dict[str, int] = {}
for split in SPLITS:
    split_sizes[split] = len(list((PROCESSED_DIR / split / 'images').glob('*')))

print('Bilder pro Split:')
for split, n in split_sizes.items():
    pct = n / sum(split_sizes.values()) * 100
    print(f'  {split:<6} {n:>5}  ({pct:.1f} %)')
print(f'  {"Total":<6} {sum(split_sizes.values()):>5}')

In [ ]:
# ── Klassenverteilung pro Split ────────────────────────────────────────────
def count_classes(split: str) -> Counter:
    counts: Counter = Counter()
    for lbl in (PROCESSED_DIR / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line.strip():
                counts[CLASS_NAMES[int(line.split()[0])]] += 1
    return counts

class_counts = {split: count_classes(split) for split in SPLITS}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Balken: Bilder pro Split
ax = axes[0]
bars = ax.bar(split_sizes.keys(), split_sizes.values(),
               color=['#4C72B0', '#DD8452', '#55A868'], width=0.5)
ax.bar_label(bars, padding=3)
ax.set_title('Bilder pro Split')
ax.set_ylabel('Anzahl Bilder')
ax.set_ylim(0, max(split_sizes.values()) * 1.15)
ax.spines[['top', 'right']].set_visible(False)

# Balken: Annotationen pro Klasse (Train)
ax = axes[1]
names  = list(CLASS_NAMES.values())
colors = [CLASS_COLORS[n] for n in names]
x = np.arange(len(names))
width = 0.25

for i, split in enumerate(SPLITS):
    vals = [class_counts[split].get(n, 0) for n in names]
    b = ax.bar(x + i * width, vals, width, label=split)
    ax.bar_label(b, padding=2, fontsize=8)

ax.set_title('Annotationen pro Klasse und Split')
ax.set_ylabel('Anzahl Annotationen')
ax.set_xticks(x + width)
ax.set_xticklabels(names)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 3 · Beispielbilder (3 × 3 Grid)

In [ ]:
def draw_yolo_boxes(ax: plt.Axes, img: Image.Image, label_path: Path) -> None:
    """Draw YOLO bounding boxes onto a matplotlib Axes."""
    ax.imshow(img)
    w, h = img.size
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            if not line.strip():
                continue
            parts = list(map(float, line.split()))
            cls_id = int(parts[0])
            cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]
            x0 = (cx - bw / 2) * w
            y0 = (cy - bh / 2) * h
            color = CLASS_COLORS[CLASS_NAMES[cls_id]]
            rect = mpatches.Rectangle(
                (x0, y0), bw * w, bh * h,
                linewidth=1.5, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x0 + 2, y0 - 4, CLASS_NAMES[cls_id],
                    color=color, fontsize=7, fontweight='bold')
    ax.axis('off')


train_images = sorted((PROCESSED_DIR / 'train' / 'images').glob('*'))
random.seed(SEED)
samples = random.sample(train_images, 9)

fig, axes = plt.subplots(3, 3, figsize=(13, 8))
fig.suptitle('9 zufällige Trainingsbilder mit Ground-Truth-Annotationen', fontsize=13)

for ax, img_path in zip(axes.flat, samples):
    lbl_path = PROCESSED_DIR / 'train' / 'labels' / img_path.with_suffix('.txt').name
    img = Image.open(img_path)
    draw_yolo_boxes(ax, img, lbl_path)
    ax.set_title(img_path.stem[:30], fontsize=7)

legend_patches = [mpatches.Patch(color=c, label=n) for n, c in CLASS_COLORS.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, frameon=False, fontsize=10)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 4 · Annotationsanalyse

In [ ]:
# ── Bounding-Box-Größen aus Trainings-Labels ───────────────────────────────
bbox_data: dict[str, list[tuple[float, float]]] = defaultdict(list)  # name → [(w, h), ...]

for lbl in (PROCESSED_DIR / 'train' / 'labels').glob('*.txt'):
    for line in lbl.read_text().strip().splitlines():
        if not line.strip():
            continue
        parts = list(map(float, line.split()))
        name = CLASS_NAMES[int(parts[0])]
        bbox_data[name].append((parts[3], parts[4]))  # normalized (w, h)

names = list(CLASS_NAMES.values())
avg_widths  = [np.mean([b[0] for b in bbox_data[n]]) for n in names]
avg_heights = [np.mean([b[1] for b in bbox_data[n]]) for n in names]
avg_areas   = [np.mean([b[0] * b[1] for b in bbox_data[n]]) for n in names]

print('Durchschnittliche normalisierte Bounding-Box-Dimensionen (Trainingsset):')
print(f'  {"Klasse":<10} {"Breite":>8} {"Höhe":>8} {"Fläche":>10}')
print('  ' + '-' * 38)
for n, w, h, a in zip(names, avg_widths, avg_heights, avg_areas):
    print(f'  {n:<10} {w:>8.4f} {h:>8.4f} {a:>10.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Ø Bounding-Box-Größe pro Klasse ───────────────────────────────────────
ax = axes[0]
x = np.arange(len(names))
width = 0.3
colors_list = [CLASS_COLORS[n] for n in names]

b1 = ax.bar(x - width / 2, avg_widths,  width, label='Ø Breite',  color=colors_list, alpha=0.9)
b2 = ax.bar(x + width / 2, avg_heights, width, label='Ø Höhe',    color=colors_list, alpha=0.5)
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=8)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('Normalisierte Größe (0–1)')
ax.set_title('Ø Bounding-Box-Größe pro Klasse\n(normalisiert auf Bildgröße)')
ax.legend(['Ø Breite', 'Ø Höhe'])
ax.spines[['top', 'right']].set_visible(False)

# ── Histogramm: Konfidenzwerte des Modells auf dem Testset ────────────────
ax = axes[1]

if MODEL_PATH.exists():
    model_tmp = YOLO(str(MODEL_PATH))
    test_images = sorted((PROCESSED_DIR / 'test' / 'images').glob('*'))
    all_confs: dict[str, list[float]] = defaultdict(list)

    for img_path in test_images:
        results = model_tmp.predict(str(img_path), device='mps', conf=0.1, verbose=False)
        boxes = results[0].boxes
        if boxes is not None and len(boxes):
            for cls_id, conf in zip(boxes.cls.tolist(), boxes.conf.tolist()):
                all_confs[CLASS_NAMES[int(cls_id)]].append(float(conf))

    bins = np.linspace(0.1, 1.0, 20)
    for name in names:
        if all_confs[name]:
            ax.hist(all_confs[name], bins=bins, alpha=0.6,
                    color=CLASS_COLORS[name], label=f'{name} (n={len(all_confs[name])})')

    ax.axvline(0.25, color='black', linestyle='--', linewidth=1, label='conf=0.25 (Threshold)')
    ax.set_xlabel('Konfidenz')
    ax.set_ylabel('Anzahl Detektionen')
    ax.set_title('Histogramm der Modell-Konfidenzwerte\n(Testset, conf ≥ 0.10)')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
else:
    ax.text(0.5, 0.5, 'models/best.pt nicht gefunden.\nTraining zuerst ausführen.',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)
    ax.set_title('Histogramm der Konfidenzwerte')

plt.tight_layout()
plt.show()

## 5 · Modell-Ergebnisse

In [ ]:
if MODEL_PATH.exists():
    model_eval = YOLO(str(MODEL_PATH))
    metrics = model_eval.val(
        data=DATASET_YAML,
        split='test',
        device='mps',
        imgsz=640,
        plots=False,
        verbose=False,
    )
    box = metrics.box

    ap50_arr  = np.array(box.ap50)
    p_arr     = np.array(box.p)
    r_arr     = np.array(box.r)
    class_idx = np.array(box.ap_class_index, dtype=int)
    cls_names_ordered = [CLASS_NAMES[i] for i in class_idx]

    print(f'Overall mAP50   : {box.map50:.4f}')
    print(f'Overall mAP50-95: {box.map:.4f}')
    print(f'Overall Precision: {box.mp:.4f}')
    print(f'Overall Recall  : {box.mr:.4f}')
    print()
    print(f'{"Klasse":<10} {"AP50":>7} {"P":>7} {"R":>7}')
    print('-' * 33)
    for i, name in enumerate(cls_names_ordered):
        print(f'{name:<10} {ap50_arr[i]:>7.4f} {p_arr[i]:>7.4f} {r_arr[i]:>7.4f}')
else:
    print('models/best.pt nicht gefunden.')
    box = None
    cls_names_ordered = list(CLASS_NAMES.values())
    ap50_arr = np.zeros(len(cls_names_ordered))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if MODEL_PATH.exists() and box is not None:
    colors_ordered = [CLASS_COLORS[n] for n in cls_names_ordered]

    # ── mAP50 pro Klasse + Overall ─────────────────────────────────────────
    ax = axes[0]
    bar_names  = cls_names_ordered + ['Overall']
    bar_values = list(ap50_arr) + [box.map50]
    bar_colors = colors_ordered + ['#888888']
    bars = ax.bar(bar_names, bar_values, color=bar_colors, width=0.5)
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('mAP50')
    ax.set_title('mAP50 pro Klasse (Testset)')
    ax.axhline(box.map50, color='#888888', linestyle='--', linewidth=1, alpha=0.6)
    ax.spines[['top', 'right']].set_visible(False)

    # ── Precision & Recall pro Klasse ──────────────────────────────────────
    ax = axes[1]
    x = np.arange(len(cls_names_ordered))
    width = 0.3
    b1 = ax.bar(x - width / 2, p_arr, width, color=colors_ordered, alpha=0.9, label='Precision')
    b2 = ax.bar(x + width / 2, r_arr, width, color=colors_ordered, alpha=0.5, label='Recall')
    ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=8)
    ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(cls_names_ordered)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score')
    ax.set_title('Precision & Recall pro Klasse (Testset)')
    ax.legend(['Precision', 'Recall'])
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 6 · Key Findings

### Datensatz

1. **Klassenimbalance:** Cracks dominieren mit ~67 % aller Annotationen (train: 1 793 von 3 341).
   Potholes und Manholes sind deutlich seltener vertreten – ein möglicher Grund für deren schwächere mAP.

2. **Unterschiedliche Box-Größen:** Cracks haben im Schnitt ~4× größere Bounding Boxes (Breite 0.157, Höhe 0.141)
   als Manholes (0.090 × 0.057) und Potholes (0.086 × 0.070). Das Modell muss also Objekte auf
   sehr unterschiedlichen Scales erkennen.

### Modell

3. **Manholes werden am besten erkannt:** mAP50 = 0.749, deutlich über Pothole (0.363) und Crack (0.356).
   Erklärung: Kanaldeckel haben eine charakteristische runde Form und gleichmäßige Textur;
   Risse und Schlaglöcher variieren stark in Form und Kontext.

4. **Konfidenzverteilung zeigt bimodales Muster:** Viele sichere Detektionen (conf > 0.7) stehen
   einer großen Menge schwacher Detektionen (conf 0.1–0.3) gegenüber – Hinweis auf grenzwertige
   Fälle (schlechte Beleuchtung, Bildunschärfe), die der Schwellenwert bei 0.25 bereits herausfiltert.

5. **Potenzial für Verbesserung:** Der Overall mAP50 von 0.489 lässt Raum nach oben. Ansätze:
   Oversampling der Minderheitsklassen, Datenaug­mentierung (Mosaic, Copy-Paste), oder
   Upgrade auf YOLOv8s/m für mehr Modellkapazität.